# Agentic AI Course — Class 1 Homework — SOLUTIONS

Reference solutions for `class1_homework.ipynb`. Instructor copy — one valid way to solve each exercise, not the only way.

For each exercise: the **core solution** replaces the original `TODO` stubs, followed by the original demo/test cell (unchanged, so you can confirm it still produces the expected output), then a **bonus cell** covering the stretch tasks.


# Class 1 — Python Refresher

## Exercise 1: Assistant Request Data

You are given a list of requests sent to an imaginary AI assistant.

### Core tasks

1. Implement `count_requests`.
2. Implement `get_unique_users`.
3. Implement `get_high_priority_requests`.
4. Implement `format_request`.
5. Handle missing dictionary keys safely.

### Stretch tasks

- Count requests by priority.
- Search messages by keyword.
- Sort requests by priority.
- Add type hints and `assert` tests.


In [ ]:
requests = [
    {"user": "Ada", "message": "What is Python?", "priority": "normal"},
    {"user": "Grace", "message": "Help! My code crashes.", "priority": "high"},
    {"user": "Ada", "message": "Explain classes.", "priority": "normal"},
    {"user": "Linus", "message": "What is a list?", "priority": "low"},
]


def count_requests(requests):
    return len(requests)


def get_unique_users(requests):
    # dict.fromkeys() preserves first-seen order while de-duplicating
    return list(dict.fromkeys(r.get("user", "Unknown") for r in requests))


def get_high_priority_requests(requests):
    return [r for r in requests if r.get("priority") == "high"]


def format_request(request):
    user = request.get("user", "Unknown")
    message = request.get("message", "")
    priority = request.get("priority", "normal")
    return f"[{priority.upper()}] {user}: {message}"


In [ ]:
print("Number of requests:", count_requests(requests))
print("Unique users:", get_unique_users(requests))
print("High-priority requests:", get_high_priority_requests(requests))

for request in requests:
    print(format_request(request))

incomplete_request = {"user": "Ada", "message": "Hello"}
print(format_request(incomplete_request))


### Bonus — Exercise 1 stretch tasks

In [ ]:
from typing import Dict, List


def count_by_priority(requests: List[dict]) -> Dict[str, int]:
    counts: Dict[str, int] = {}
    for r in requests:
        p = r.get("priority", "normal")
        counts[p] = counts.get(p, 0) + 1
    return counts


def search_by_keyword(requests: List[dict], keyword: str) -> List[dict]:
    keyword_lower = keyword.lower()
    return [r for r in requests if keyword_lower in r.get("message", "").lower()]


def sort_by_priority(requests: List[dict]) -> List[dict]:
    # Explicit order since "high" should sort before "low" alphabetically fails us
    order = {"high": 0, "normal": 1, "low": 2}
    return sorted(requests, key=lambda r: order.get(r.get("priority", "normal"), 99))


print(count_by_priority(requests))
print(search_by_keyword(requests, "class"))
print([r["user"] for r in sort_by_priority(requests)])

assert count_by_priority(requests) == {"normal": 2, "high": 1, "low": 1}
assert search_by_keyword(requests, "list") == [requests[3]]
assert [r["priority"] for r in sort_by_priority(requests)] == ["high", "normal", "normal", "low"]
print("Exercise 1 bonus tests passed!")


## Exercise 2: Build a `Message` Class

LLM applications commonly use messages with roles such as:

- `system`
- `user`
- `assistant`

### Core tasks

1. Store `role` and `content`.
2. Validate roles.
3. Add `display()`.
4. Add `word_count()`.
5. Add `is_empty()`.

### Stretch tasks

- Add `contains_keyword(keyword)`.
- Implement `__str__`.
- Normalize roles with `.strip().lower()`.
- Add metadata or a timestamp.


In [ ]:
class Message:
    ALLOWED_ROLES = {"system", "user", "assistant"}

    def __init__(self, role, content):
        if role not in self.ALLOWED_ROLES:
            raise ValueError(
                f"Invalid role: {role!r}. Must be one of {sorted(self.ALLOWED_ROLES)}."
            )
        self.role = role
        self.content = content

    def display(self):
        return f"{self.role.upper()}: {self.content}"

    def word_count(self):
        return len(self.content.split())

    def is_empty(self):
        return len(self.content.strip()) == 0


In [ ]:
message = Message("user", "Explain Python classes.")

print(message.display())
print("Word count:", message.word_count())
print("Is empty:", message.is_empty())

# Try after role validation is implemented:
# invalid_message = Message("developer", "Hello")


### Bonus — Exercise 2 stretch tasks

In [ ]:
class Message:
    ALLOWED_ROLES = {"system", "user", "assistant"}

    def __init__(self, role, content, timestamp=None):
        role = role.strip().lower()  # normalize before validating
        if role not in self.ALLOWED_ROLES:
            raise ValueError(
                f"Invalid role: {role!r}. Must be one of {sorted(self.ALLOWED_ROLES)}."
            )
        self.role = role
        self.content = content
        self.timestamp = timestamp  # e.g. datetime.now(), left as plain metadata here

    def display(self):
        return f"{self.role.upper()}: {self.content}"

    def word_count(self):
        return len(self.content.split())

    def is_empty(self):
        return len(self.content.strip()) == 0

    def contains_keyword(self, keyword):
        return keyword.lower() in self.content.lower()

    def __str__(self):
        return self.display()


m = Message("  User  ", "Explain Python classes.")
assert m.role == "user"  # normalized
assert str(m) == "USER: Explain Python classes."
assert m.contains_keyword("python") is True
assert m.contains_keyword("java") is False

try:
    Message("developer", "Hello")
    assert False, "should have raised ValueError"
except ValueError:
    pass

print("Exercise 2 bonus tests passed!")


## Exercise 3: Build a `Conversation` Class

A conversation contains multiple `Message` objects.

### Core tasks

1. Store messages in a list.
2. Implement `add_message(message)`.
3. Implement `message_count()`.
4. Implement `transcript()`.
5. Implement `latest_message()`.
6. Implement `count_by_role(role)`.

### Stretch tasks

- Add `get_messages_by_role(role)`.
- Add `clear()`.
- Limit the maximum number of stored messages.
- Validate that only `Message` objects can be added.


In [ ]:
class Message:
    ALLOWED_ROLES = {"system", "user", "assistant"}

    def __init__(self, role, content):
        if role not in self.ALLOWED_ROLES:
            raise ValueError(
                f"Invalid role: {role!r}. Must be one of {sorted(self.ALLOWED_ROLES)}."
            )
        self.role = role
        self.content = content

    def display(self):
        return f"{self.role.upper()}: {self.content}"

    def word_count(self):
        return len(self.content.split())

    def is_empty(self):
        return len(self.content.strip()) == 0


class Conversation:
    def __init__(self):
        self.messages = []

    def add_message(self, message):
        self.messages.append(message)

    def message_count(self):
        return len(self.messages)

    def transcript(self):
        return "\n".join(m.display() for m in self.messages)

    def latest_message(self):
        if not self.messages:
            return None
        return self.messages[-1]

    def count_by_role(self, role):
        return len([m for m in self.messages if m.role == role])


In [ ]:
conversation = Conversation()

conversation.add_message(Message("user", "What is OOP?"))
conversation.add_message(
    Message("assistant", "OOP organizes programs using objects.")
)

print("Message count:", conversation.message_count())
print("Latest message:", conversation.latest_message().display())
print("User messages:", conversation.count_by_role("user"))
print()
print(conversation.transcript())


### Bonus — Exercise 3 stretch tasks

In [ ]:
class Conversation:
    def __init__(self, max_messages=None):
        self.messages = []
        self.max_messages = max_messages  # None = unlimited

    def add_message(self, message):
        if not isinstance(message, Message):
            raise TypeError("Conversation can only store Message objects.")
        if self.max_messages is not None and len(self.messages) >= self.max_messages:
            self.messages.pop(0)  # drop the oldest to make room
        self.messages.append(message)

    def message_count(self):
        return len(self.messages)

    def transcript(self):
        return "\n".join(m.display() for m in self.messages)

    def latest_message(self):
        return self.messages[-1] if self.messages else None

    def count_by_role(self, role):
        return len([m for m in self.messages if m.role == role])

    def get_messages_by_role(self, role):
        return [m for m in self.messages if m.role == role]

    def clear(self):
        self.messages = []


c = Conversation(max_messages=2)
c.add_message(Message("user", "one"))
c.add_message(Message("user", "two"))
c.add_message(Message("user", "three"))  # should push out "one"
assert c.message_count() == 2
assert [m.content for m in c.messages] == ["two", "three"]

try:
    c.add_message("not a message")
    assert False, "should have raised TypeError"
except TypeError:
    pass

c.clear()
assert c.message_count() == 0
print("Exercise 3 bonus tests passed!")


## Exercise 4: Build Tool Classes

Agents use tools to perform actions. These tools are local and do not use APIs.

### Core tasks

1. Complete `CalculatorTool`.
2. Support addition such as `4 + 7`.
3. Return a helpful error message for invalid input.
4. Complete `TextStatisticsTool`.
5. Use both tools through the shared `run()` method.

> Do **not** use `eval()`. It can execute arbitrary Python code and is unsafe for user input.

### Stretch tasks

- Support `-`, `*`, and `/`.
- Add a mock weather tool.
- Create a `ToolResult` class.


In [ ]:
class Tool:
    def __init__(self, name, description):
        self.name = name
        self.description = description

    def run(self, input_text):
        raise NotImplementedError("Each tool must implement run().")


class CalculatorTool(Tool):
    def __init__(self):
        super().__init__("calculator", "Adds two numbers, e.g. '4 + 7'.")

    def run(self, input_text):
        parts = input_text.split()
        if len(parts) != 3 or parts[1] != "+":
            return "Error: expected format '<number> + <number>', e.g. '4 + 7'."
        left_str, _op, right_str = parts
        try:
            left, right = float(left_str), float(right_str)
        except ValueError:
            return f"Error: '{left_str}' and '{right_str}' must be numbers."
        result = left + right
        if result == int(result):
            result = int(result)
        return str(result)


class TextStatisticsTool(Tool):
    def __init__(self):
        super().__init__("text_statistics", "Counts characters and words in a piece of text.")

    def run(self, input_text):
        char_count = len(input_text)
        word_count = len(input_text.split())
        return f"{word_count} words, {char_count} characters"


In [ ]:
tools = [CalculatorTool(), TextStatisticsTool()]

for tool in tools:
    print(f"{tool.name}: {tool.description}")

calculator = CalculatorTool()
statistics = TextStatisticsTool()

print(calculator.run("4 + 7"))
print(statistics.run("Python is useful for AI apps"))


### Bonus — Exercise 4 stretch tasks

In [ ]:
class ToolResult:
    def __init__(self, success, value):
        self.success = success
        self.value = value

    def __str__(self):
        prefix = "OK" if self.success else "ERROR"
        return f"[{prefix}] {self.value}"


class CalculatorTool(Tool):
    OPERATORS = {"+", "-", "*", "/"}

    def __init__(self):
        super().__init__("calculator", "Evaluates '<number> <op> <number>' with +, -, *, /.")

    def run(self, input_text):
        parts = input_text.split()
        if len(parts) != 3 or parts[1] not in self.OPERATORS:
            return str(ToolResult(False, "expected '<number> <op> <number>', op in + - * /"))
        left_str, op, right_str = parts
        try:
            left, right = float(left_str), float(right_str)
        except ValueError:
            return str(ToolResult(False, f"'{left_str}' and '{right_str}' must be numbers"))

        if op == "+":
            result = left + right
        elif op == "-":
            result = left - right
        elif op == "*":
            result = left * right
        else:  # "/"
            if right == 0:
                return str(ToolResult(False, "division by zero"))
            result = left / right

        if result == int(result):
            result = int(result)
        return str(ToolResult(True, result))


class MockWeatherTool(Tool):
    FAKE_WEATHER = {"aarhus": "14°C, cloudy", "copenhagen": "13°C, windy"}

    def __init__(self):
        super().__init__("weather", "Looks up (fake) current weather for a city.")

    def run(self, input_text):
        city = input_text.strip().lower()
        return self.FAKE_WEATHER.get(city, f"No weather data for '{input_text}'.")


calc = CalculatorTool()
assert calc.run("4 + 7") == "[OK] 11"
assert calc.run("10 / 0") == "[ERROR] division by zero"
weather = MockWeatherTool()
assert weather.run("Aarhus") == "14°C, cloudy"
print("Exercise 4 bonus tests passed!")


## Exercise 5: Build a Rule-Based Assistant

This is **not** an LLM. It uses rules to choose a tool.

### Expected behaviour

```text
User: calculate 8 + 12
Assistant: Calculator result: 20
```

### Core tasks

1. Store user input in the conversation.
2. Implement `get_tool(name)`.
3. Route `calculate ` commands to the calculator.
4. Route `count words in ` commands to the statistics tool.
5. Return a help message for unknown input.
6. Store assistant responses in the conversation.

### Stretch tasks

- Add `help`.
- Add `history`.
- Match commands case-insensitively.
- Add `calc ` as an alias.


In [ ]:
class CalculatorTool(Tool):
    def __init__(self):
        super().__init__("calculator", "Adds two numbers, e.g. '4 + 7'.")

    def run(self, input_text):
        parts = input_text.split()
        if len(parts) != 3 or parts[1] != "+":
            return "Error: expected format '<number> + <number>', e.g. '4 + 7'."
        left_str, _op, right_str = parts
        try:
            left, right = float(left_str), float(right_str)
        except ValueError:
            return f"Error: '{left_str}' and '{right_str}' must be numbers."
        result = left + right
        if result == int(result):
            result = int(result)
        return str(result)


class SimpleAssistant:
    def __init__(self, tools):
        self.tools = tools
        self.conversation = Conversation()

    def get_tool(self, name):
        for tool in self.tools:
            if tool.name == name:
                return tool
        return None

    def respond(self, user_input):
        self.conversation.add_message(Message("user", user_input))

        if user_input.startswith("calculate "):
            expression = user_input[len("calculate "):]
            response = self.get_tool("calculator").run(expression)
            response = f"Calculator result: {response}"
        elif user_input.startswith("count words in "):
            text = user_input[len("count words in "):]
            response = self.get_tool("text_statistics").run(text)
        else:
            response = "Sorry, I didn't understand that. Try 'calculate <expr>' or 'count words in <text>'."

        self.conversation.add_message(Message("assistant", response))
        return response


In [ ]:
assistant = SimpleAssistant(
    tools=[
        CalculatorTool(),
        TextStatisticsTool(),
    ]
)

print(assistant.respond("calculate 8 + 12"))
print(assistant.respond("count words in Agentic systems use tools"))
print(assistant.respond("Hello there"))

print()
print("Conversation transcript:")
print(assistant.conversation.transcript())


### Bonus — Exercise 5 stretch tasks

In [ ]:
class SimpleAssistant:
    HELP_TEXT = (
        "Try one of:\n"
        "  calculate <a> + <b>\n"
        "  calc <a> + <b>\n"
        "  count words in <text>\n"
        "  history\n"
        "  help"
    )

    def __init__(self, tools):
        self.tools = tools
        self.conversation = Conversation()

    def get_tool(self, name):
        for tool in self.tools:
            if tool.name == name:
                return tool
        return None

    def respond(self, user_input):
        self.conversation.add_message(Message("user", user_input))
        lowered = user_input.strip().lower()

        if lowered in ("help", "?"):
            response = self.HELP_TEXT
        elif lowered == "history":
            response = self.conversation.transcript()
        elif lowered.startswith("calculate ") or lowered.startswith("calc "):
            prefix_len = len("calculate ") if lowered.startswith("calculate ") else len("calc ")
            expression = user_input[prefix_len:]
            response = f"Calculator result: {self.get_tool('calculator').run(expression)}"
        elif lowered.startswith("count words in "):
            text = user_input[len("count words in "):]
            response = self.get_tool("text_statistics").run(text)
        else:
            response = "Sorry, I didn't understand that. Type 'help' for options."

        self.conversation.add_message(Message("assistant", response))
        return response


a = SimpleAssistant([CalculatorTool(), TextStatisticsTool()])
assert a.respond("CALC 2 + 2") == "Calculator result: 4"
assert a.respond("help") == SimpleAssistant.HELP_TEXT
print("Exercise 5 bonus tests passed!")


## Exercise 6: Basic Tests

Write `assert` statements to test your code.

Suggested tests:

- The calculator adds two numbers correctly.
- Invalid calculator input returns an error.
- A message counts words correctly.
- A conversation stores messages.
- The assistant handles a calculator request.


In [ ]:
calculator = CalculatorTool()
assert calculator.run("2 + 3") == "5"
assert "Error" in calculator.run("not valid")

message = Message("user", "Hello world")
assert message.word_count() == 2
assert message.is_empty() is False

conversation = Conversation()
conversation.add_message(Message("user", "Hello"))
assert conversation.message_count() == 1

assistant = SimpleAssistant([CalculatorTool(), TextStatisticsTool()])
assert assistant.respond("calculate 10 + 5") == "Calculator result: 15"

print("All tests passed!")


# Reflection

1. What is the difference between a class and an object?
2. When is composition useful?
3. What does inheritance allow us to do with tools?
4. Why should we avoid `eval()` for user input?
5. How could this assistant later be enhanced using an LLM?

**Suggested talking points (instructor reference, not a rubric):**

1. A class is the blueprint (`Message`, `Tool`); an object is one specific instance created from it (a particular `Message("user", "hi")`).
2. Composition is useful when something *has* other things with their own behavior — `Conversation` has `Message`s, `SimpleAssistant` has `Tool`s — rather than trying to cram everything into one inheritance chain.
3. Inheritance lets every tool share the same interface (`Tool.run()`), so `SimpleAssistant` can call `.run()` on any tool without knowing which specific subclass it is — this is polymorphism in action.
4. `eval()` executes arbitrary Python — user input like `__import__('os').system('rm -rf /')` would run with the same permissions as your program. Parsing input manually (as the `CalculatorTool` does) avoids this entirely.
5. An LLM could replace the rule-based routing in `SimpleAssistant.respond()` — instead of matching string prefixes like `"calculate "`, the LLM could decide which tool to call and with what arguments (this is exactly what "tool use" / "function calling" means, coming up in Class 4–5).
